## <h1><center>POYO Single-Session Decoding on DANDI:000688</center></h1>

<center><b>CS 4782 Final Project Reproduction Notebook</b></center>

&nbsp;

---

**Goal:** Reproduce a single-session POYO velocity decoding experiment from *A Unified, Scalable Framework for Neural Population Decoding* using one automatically selected session from DANDI `000688`, then compare POYO against a Wiener filter, MLP, and GRU.

**Success means:** the notebook runs end-to-end from a fresh Colab runtime, inspects the NWB structure before using fields, trains all four models, reports `R^2` on held-out data, renders the required figures, and saves artifacts under `./artifacts/`.


# Part 0: Requirements Checklist

This notebook was grounded against the local course materials before implementation:

- `DL Final Project proposal.pdf`: target reproduction is single-session POYO velocity decoding with Wiener / MLP / GRU baselines; the proposal cites a stretch target near `R^2 ≈ 0.97`.
- `CS 4782 Final Project Instructions.pdf`: emphasize clear methodology, visual results, discussion of discrepancies, and reproducibility.
- `POYO1.pdf`: use a `1s` context window, `R^2` as the primary metric, and trial-based `20%` test / `10%` validation splits when trials exist.
- `Assignment 2` and `Assignment 3`: follow the same narrative style with setup first, clear sectioning, and visuals throughout.

Implementation checklist satisfied here:

- Auto-select the smallest valid published session from DANDI `000688`.
- Inspect NWB structure before touching any field names.
- Train and evaluate Wiener, MLP, GRU, and POYO end-to-end.
- Save figures plus `metrics.json` under `./artifacts/`.
- Compare results to the paper and explain any mismatch.

Important note:

- Because this notebook auto-selects the smallest valid session, it may land on a random-target session rather than a center-out session. The proposal's `~0.97` target is more consistent with easier center-out conditions, while the paper's Section 3.2 reports a lower average for random-target sessions. We will call that out explicitly in the final discussion.


In [ ]:
# Top-of-notebook configuration cell.

FAST_DEV_RUN = True
RANDOM_SEED = 42

CONTEXT_SEC = 1.0
BIN_SIZE_SEC = 0.01
TARGET_STRIDE_SEC = 0.10 if FAST_DEV_RUN else 0.05

MAX_CANDIDATES_TO_CHECK = 5
MAX_EPOCHS_MLP = 3 if FAST_DEV_RUN else 20
MAX_EPOCHS_GRU = 3 if FAST_DEV_RUN else 20
MAX_EPOCHS_POYO = 2 if FAST_DEV_RUN else 10

MLP_BATCH_SIZE = 256
GRU_BATCH_SIZE = 128
POYO_BATCH_SIZE = 48 if FAST_DEV_RUN else 64

POYO_LR = 1e-3
POYO_WEIGHT_DECAY = 1e-4

FAST_DEV_MAX_SAMPLES = {"train": 1200, "val": 300, "test": 400}


# Part 1: Environment Setup

In [ ]:
%pip -q install dandi pynwb temporaldata matplotlib pandas scikit-learn nwbwidgets \
    'torchmetrics>=1.6.0' einops==0.6.1 hydra-core==1.3.2 torchtyping==0.1.5 rich pytorch_brain

from pathlib import Path
import subprocess

Path("third_party").mkdir(exist_ok=True)
if not Path("third_party/torch_brain").exists():
    !git clone --depth 1 https://github.com/neuro-galaxy/torch_brain.git third_party/torch_brain

clone_commit = subprocess.check_output(
    ["git", "-C", "third_party/torch_brain", "rev-parse", "HEAD"],
    text=True,
).strip()
print("torch_brain clone commit:", clone_commit)


In [ ]:
import os
import platform
import numpy as np
import pandas as pd
import torch
import torch_brain

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("torch_brain:", getattr(torch_brain, "__version__", "unknown"))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch.use_deterministic_algorithms(False)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


In [ ]:
import inspect
import subprocess
from torch_brain.models import POYO
from torch_brain.registry import MODALITY_REGISTRY

print("Verified POYO signature:")
print(inspect.signature(POYO))
print("\nVerified cursor readout spec:")
print(MODALITY_REGISTRY["cursor_velocity_2d"])
print("\nReference files in the official clone:")
print(
    subprocess.check_output(
        ["bash", "-lc", "find third_party/torch_brain/examples/poyo -maxdepth 2 -type f | sort"],
        text=True,
    )
)


# Part 2: Shared Support Code

In [ ]:
from __future__ import annotations

import copy
import json
import math
import random
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch_brain
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from temporaldata import ArrayDict, Interval, IrregularTimeSeries
from torch.utils.data import DataLoader, Dataset, TensorDataset

from torch_brain.data import collate as tb_collate
from torch_brain.models import POYO
from torch_brain.optim import SparseLamb
from torch_brain.registry import MODALITY_REGISTRY


PROJECT_ROOT = Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
TMP_DIR = PROJECT_ROOT / "tmp" / "dandi_inspect"
ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)
TMP_DIR.mkdir(exist_ok=True, parents=True)


FAST_DEV_RUN = globals().get("FAST_DEV_RUN", True)
RANDOM_SEED = globals().get("RANDOM_SEED", 42)
CONTEXT_SEC = globals().get("CONTEXT_SEC", 1.0)
BIN_SIZE_SEC = globals().get("BIN_SIZE_SEC", 0.01)
TARGET_STRIDE_SEC = globals().get("TARGET_STRIDE_SEC", 0.10 if FAST_DEV_RUN else 0.05)
MAX_CANDIDATES_TO_CHECK = globals().get("MAX_CANDIDATES_TO_CHECK", 5)
MAX_EPOCHS_MLP = globals().get("MAX_EPOCHS_MLP", 3 if FAST_DEV_RUN else 20)
MAX_EPOCHS_GRU = globals().get("MAX_EPOCHS_GRU", 3 if FAST_DEV_RUN else 20)
MAX_EPOCHS_POYO = globals().get("MAX_EPOCHS_POYO", 2 if FAST_DEV_RUN else 10)
MLP_BATCH_SIZE = globals().get("MLP_BATCH_SIZE", 256)
GRU_BATCH_SIZE = globals().get("GRU_BATCH_SIZE", 128)
POYO_BATCH_SIZE = globals().get("POYO_BATCH_SIZE", 48 if FAST_DEV_RUN else 64)
POYO_LR = globals().get("POYO_LR", 1e-3)
POYO_WEIGHT_DECAY = globals().get("POYO_WEIGHT_DECAY", 1e-4)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FAST_DEV_MAX_SAMPLES = globals().get("FAST_DEV_MAX_SAMPLES", {"train": 1200, "val": 300, "test": 400})


@dataclass
class SimpleSessionDescription:
    id: str


class MiniData:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

    def get_nested_attribute(self, dotted_key: str):
        obj = self
        for part in dotted_key.split("."):
            obj = getattr(obj, part)
        return obj


@dataclass
class SelectedAsset:
    dandiset_id: str
    version_id: str
    asset_id: str
    path: str
    size_bytes: int
    local_path: Path
    subject_id: str | None
    session_description: str
    duration_sec: float
    units_count: int
    trials_count: int


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_default(obj):
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Unsupported type: {type(obj)}")


def latest_published_dandiset(dandiset_id: str = "000688"):
    with DandiAPIClient() as client:
        ds = client.get_dandiset(dandiset_id)
        return ds


def download_smallest_valid_asset(
    dandiset_id: str = "000688",
    candidate_limit: int = MAX_CANDIDATES_TO_CHECK,
) -> SelectedAsset:
    ds = latest_published_dandiset(dandiset_id)
    candidate_rows: List[Tuple[int, object]] = []
    for asset in ds.get_assets(order="path"):
        if asset.path.endswith(".nwb"):
            candidate_rows.append((asset.size, asset))
    candidate_rows.sort(key=lambda x: x[0])

    for size_bytes, asset in candidate_rows[:candidate_limit]:
        local_path = TMP_DIR / Path(asset.path).name
        if not local_path.exists() or local_path.stat().st_size == 0:
            print(f"Downloading candidate: {asset.path} ({size_bytes / 1e6:.2f} MB)")
            asset.download(local_path)

        try:
            with NWBHDF5IO(str(local_path), "r", load_namespaces=True) as io:
                nwb = io.read()
                units = getattr(nwb, "units", None)
                units_count = 0 if units is None else len(units.id[:])
                trials = getattr(nwb, "trials", None)
                trials_count = 0 if trials is None else len(trials.id[:])
                behavior = nwb.processing.get("behavior", None)
                if behavior is None:
                    continue
                velocity_iface = behavior.data_interfaces.get("Velocity", None)
                position_iface = behavior.data_interfaces.get("Position", None)
                has_velocity = (
                    velocity_iface is not None
                    and hasattr(velocity_iface, "time_series")
                    and "cursor_vel" in velocity_iface.time_series
                )
                has_position = (
                    position_iface is not None
                    and hasattr(position_iface, "spatial_series")
                    and "cursor_pos" in position_iface.spatial_series
                )
                if not (units_count > 0 and trials_count >= 20 and (has_velocity or has_position)):
                    continue
                if has_velocity:
                    vel = velocity_iface.time_series["cursor_vel"]
                    ts = np.asarray(vel.timestamps[:])
                    duration_sec = float(ts[-1] - ts[0]) if len(ts) else 0.0
                else:
                    pos = position_iface.spatial_series["cursor_pos"]
                    ts = np.asarray(pos.timestamps[:])
                    duration_sec = float(ts[-1] - ts[0]) if len(ts) else 0.0
                if duration_sec < 120.0:
                    continue
                return SelectedAsset(
                    dandiset_id=dandiset_id,
                    version_id=ds.version_id,
                    asset_id=asset.identifier,
                    path=asset.path,
                    size_bytes=size_bytes,
                    local_path=local_path,
                    subject_id=getattr(getattr(nwb, "subject", None), "subject_id", None),
                    session_description=nwb.session_description,
                    duration_sec=duration_sec,
                    units_count=units_count,
                    trials_count=trials_count,
                )
        except Exception as exc:
            print(f"Skipping {asset.path}: {exc!r}")
            continue

    raise RuntimeError("Could not find a valid asset among the inspected candidates.")


def inspect_nwb_file(local_path: Path) -> Dict:
    with NWBHDF5IO(str(local_path), "r", load_namespaces=True) as io:
        nwb = io.read()
        structure = {
            "acquisition_keys": list(nwb.acquisition.keys()),
            "processing_keys": list(nwb.processing.keys()),
            "interval_keys": list(nwb.intervals.keys()) if nwb.intervals is not None else [],
            "units_count": 0 if nwb.units is None else len(nwb.units.id[:]),
            "unit_columns": [] if nwb.units is None else list(nwb.units.colnames),
            "behavior_interfaces": {},
        }
        if "behavior" in nwb.processing:
            for name, obj in nwb.processing["behavior"].data_interfaces.items():
                if hasattr(obj, "time_series"):
                    children = list(obj.time_series.keys())
                elif hasattr(obj, "spatial_series"):
                    children = list(obj.spatial_series.keys())
                else:
                    children = []
                structure["behavior_interfaces"][name] = children
        return structure


def load_session_arrays(local_path: Path) -> Dict:
    with NWBHDF5IO(str(local_path), "r", load_namespaces=True) as io:
        nwb = io.read()
        behavior = nwb.processing["behavior"]
        vel = behavior.data_interfaces["Velocity"].time_series["cursor_vel"]
        pos = behavior.data_interfaces["Position"].spatial_series["cursor_pos"]
        trials = nwb.trials.to_dataframe().copy()
        units = nwb.units
        unit_ids = np.array([str(x) for x in units.id[:]], dtype=object)
        spike_times_by_unit = [np.asarray(st, dtype=np.float64) for st in units["spike_times"][:]]
        return {
            "vel_timestamps": np.asarray(vel.timestamps[:], dtype=np.float64),
            "vel_values": np.asarray(vel.data[:], dtype=np.float32),
            "pos_values": np.asarray(pos.data[:], dtype=np.float32),
            "trials": trials,
            "unit_ids": unit_ids,
            "spike_times_by_unit": spike_times_by_unit,
            "subject_id": getattr(getattr(nwb, "subject", None), "subject_id", None),
            "session_description": nwb.session_description,
        }


def choose_successful_trials(trials: pd.DataFrame, context_sec: float, min_trials: int = 20) -> pd.DataFrame:
    trial_df = trials.copy()
    duration = trial_df["stop_time"] - trial_df["start_time"]
    trial_df = trial_df.loc[duration >= context_sec + 0.05].copy()
    if "result" in trial_df.columns and (trial_df["result"] == "R").sum() >= min_trials:
        trial_df = trial_df.loc[trial_df["result"] == "R"].copy()
    trial_df["duration_sec"] = trial_df["stop_time"] - trial_df["start_time"]
    return trial_df


def split_trials(trial_df: pd.DataFrame, seed: int) -> Dict[str, np.ndarray]:
    ids = trial_df.index.to_numpy()
    rng = np.random.default_rng(seed)
    shuffled = ids.copy()
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_test = max(1, round(0.20 * n))
    n_val = max(1, round(0.10 * n))
    test_ids = np.sort(shuffled[:n_test])
    val_ids = np.sort(shuffled[n_test : n_test + n_val])
    train_ids = np.sort(shuffled[n_test + n_val :])
    return {"train": train_ids, "val": val_ids, "test": test_ids}


def build_sample_table(
    vel_timestamps: np.ndarray,
    vel_values: np.ndarray,
    trials: pd.DataFrame,
    split_map: Dict[str, np.ndarray],
    context_sec: float,
    stride_sec: float,
) -> pd.DataFrame:
    dt = float(np.median(np.diff(vel_timestamps)))
    stride_steps = max(1, int(round(stride_sec / dt)))
    rows = []
    trial_index_lookup = set(trials.index.to_numpy())
    for split_name, trial_ids in split_map.items():
        for trial_id in trial_ids:
            if trial_id not in trial_index_lookup:
                continue
            row = trials.loc[trial_id]
            mask = (vel_timestamps >= row["start_time"] + context_sec) & (vel_timestamps <= row["stop_time"])
            idx = np.flatnonzero(mask)
            if len(idx) == 0:
                continue
            idx = idx[::stride_steps]
            for vel_idx in idx:
                rows.append(
                    {
                        "split": split_name,
                        "trial_id": int(trial_id),
                        "time": float(vel_timestamps[vel_idx]),
                        "vel_idx": int(vel_idx),
                        "vx": float(vel_values[vel_idx, 0]),
                        "vy": float(vel_values[vel_idx, 1]),
                    }
                )
    sample_df = pd.DataFrame(rows).sort_values(["split", "trial_id", "time"]).reset_index(drop=True)
    return sample_df


def bin_full_session(
    spike_times_by_unit: List[np.ndarray],
    end_time: float,
    bin_size_sec: float,
) -> np.ndarray:
    n_units = len(spike_times_by_unit)
    n_bins = int(math.floor(end_time / bin_size_sec)) + 1
    counts = np.zeros((n_bins, n_units), dtype=np.float32)
    for unit_idx, spike_times in enumerate(spike_times_by_unit):
        bin_idx = np.floor(spike_times / bin_size_sec).astype(int)
        valid = (bin_idx >= 0) & (bin_idx < n_bins)
        if valid.any():
            np.add.at(counts[:, unit_idx], bin_idx[valid], 1.0)
    return counts


def build_baseline_arrays(
    sample_df: pd.DataFrame,
    binned_counts: np.ndarray,
    context_bins: int,
) -> Tuple[np.ndarray, np.ndarray]:
    vel_idx = sample_df["vel_idx"].to_numpy(dtype=int)
    feature_idx = vel_idx[:, None] - context_bins + np.arange(context_bins)
    X = binned_counts[feature_idx]
    y = sample_df[["vx", "vy"]].to_numpy(dtype=np.float32)
    return X.astype(np.float32), y


def r2_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "r2_mean": float(r2_score(y_true, y_pred, multioutput="uniform_average")),
        "r2_vx": float(r2_score(y_true[:, 0], y_pred[:, 0])),
        "r2_vy": float(r2_score(y_true[:, 1], y_pred[:, 1])),
    }


class MLPDecoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        return self.net(x)


class GRUDecoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.readout = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.readout(out[:, -1])


def iterate_batches(
    loader: DataLoader,
    model: nn.Module,
    optimizer=None,
    target_mean: np.ndarray | None = None,
    target_std: np.ndarray | None = None,
):
    criterion = nn.MSELoss()
    is_train = optimizer is not None
    total_loss = 0.0
    preds, targets = [], []
    model.train(is_train)
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        with torch.set_grad_enabled(is_train):
            pred = model(xb)
            loss = criterion(pred, yb)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * xb.size(0)
        pred_np = pred.detach().cpu().numpy()
        target_np = yb.detach().cpu().numpy()
        if target_mean is not None and target_std is not None:
            pred_np = pred_np * target_std + target_mean
            target_np = target_np * target_std + target_mean
        preds.append(pred_np)
        targets.append(target_np)
    preds_np = np.concatenate(preds, axis=0)
    targets_np = np.concatenate(targets, axis=0)
    metrics = r2_metrics(targets_np, preds_np)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics, preds_np, targets_np


def train_torch_decoder(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    max_epochs: int,
    lr: float = 1e-3,
    target_mean: np.ndarray | None = None,
    target_std: np.ndarray | None = None,
) -> Tuple[nn.Module, List[Dict]]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    history = []
    best_state = None
    best_val = -np.inf
    model.to(DEVICE)
    for epoch in range(1, max_epochs + 1):
        train_metrics, _, _ = iterate_batches(
            train_loader,
            model,
            optimizer,
            target_mean=target_mean,
            target_std=target_std,
        )
        val_metrics, _, _ = iterate_batches(
            val_loader,
            model,
            optimizer=None,
            target_mean=target_mean,
            target_std=target_std,
        )
        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "val_loss": val_metrics["loss"],
            "train_r2_mean": train_metrics["r2_mean"],
            "val_r2_mean": val_metrics["r2_mean"],
        }
        history.append(row)
        if val_metrics["r2_mean"] > best_val:
            best_val = val_metrics["r2_mean"]
            best_state = copy.deepcopy(model.state_dict())
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


class POYOSingleQueryDataset(Dataset):
    def __init__(
        self,
        sample_df: pd.DataFrame,
        spike_times_by_unit: List[np.ndarray],
        unit_ids: np.ndarray,
        session_id: str,
        model: POYO,
        context_sec: float = CONTEXT_SEC,
        normalize_std: float = 20.0,
    ):
        self.sample_df = sample_df.reset_index(drop=True)
        self.spike_times_by_unit = spike_times_by_unit
        self.unit_ids = unit_ids
        self.session_id = session_id
        self.model = model
        self.context_sec = context_sec
        self.normalize_std = normalize_std

    def __len__(self):
        return len(self.sample_df)

    def __getitem__(self, idx):
        row = self.sample_df.iloc[idx]
        end = float(row["time"])
        start = end - self.context_sec
        spike_timestamps = []
        spike_unit_index = []
        for unit_idx, spike_times in enumerate(self.spike_times_by_unit):
            left = np.searchsorted(spike_times, start, side="left")
            right = np.searchsorted(spike_times, end, side="right")
            if right > left:
                rel = spike_times[left:right] - start
                spike_timestamps.append(rel.astype(np.float64))
                spike_unit_index.append(np.full(len(rel), unit_idx, dtype=np.int64))
        if spike_timestamps:
            spike_timestamps_arr = np.concatenate(spike_timestamps)
            spike_unit_index_arr = np.concatenate(spike_unit_index)
            order = np.argsort(spike_timestamps_arr, kind="stable")
            spike_timestamps_arr = spike_timestamps_arr[order]
            spike_unit_index_arr = spike_unit_index_arr[order]
        else:
            spike_timestamps_arr = np.empty((0,), dtype=np.float64)
            spike_unit_index_arr = np.empty((0,), dtype=np.int64)

        data = MiniData(
            spikes=IrregularTimeSeries(
                timestamps=spike_timestamps_arr,
                unit_index=spike_unit_index_arr,
                domain="auto",
            ),
            cursor=IrregularTimeSeries(
                timestamps=np.array([self.context_sec], dtype=np.float64),
                vel=np.array([[row["vx"], row["vy"]]], dtype=np.float32),
                domain="auto",
            ),
            units=ArrayDict(id=self.unit_ids.copy()),
            session=SimpleSessionDescription(id=self.session_id),
            config={
                "readout": {
                    "readout_id": "cursor_velocity_2d",
                    "normalize_mean": 0.0,
                    "normalize_std": self.normalize_std,
                }
            },
            absolute_start=float(start),
            domain=Interval(0.0, self.context_sec),
        )
        return self.model.tokenize(data)


def evaluate_poyo_loader(model: POYO, loader: DataLoader, denorm_std: float = 20.0):
    model.eval()
    preds, targets = [], []
    total_loss = 0.0
    criterion = nn.MSELoss(reduction="none")
    with torch.no_grad():
        for batch in loader:
            model_inputs = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch["model_inputs"].items()}
            target_values = batch["target_values"].to(DEVICE)
            target_weights = batch["target_weights"].to(DEVICE)
            output_mask = batch["model_inputs"]["output_mask"].to(DEVICE)
            output = model(**model_inputs)
            masked_output = output[output_mask]
            masked_target = target_values[output_mask]
            masked_weights = target_weights[output_mask]
            loss = criterion(masked_output, masked_target)
            loss = (loss * masked_weights.unsqueeze(-1)).mean()
            total_loss += loss.item() * target_values.shape[0]
            preds.append((masked_output.cpu().numpy() * denorm_std).reshape(-1, 2))
            targets.append((masked_target.cpu().numpy() * denorm_std).reshape(-1, 2))
    preds_np = np.concatenate(preds, axis=0)
    targets_np = np.concatenate(targets, axis=0)
    metrics = r2_metrics(targets_np, preds_np)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics, preds_np, targets_np


def train_poyo_model(
    model: POYO,
    train_loader: DataLoader,
    val_loader: DataLoader,
    max_epochs: int,
) -> Tuple[POYO, List[Dict]]:
    model.to(DEVICE)
    special_emb_params = list(model.unit_emb.parameters()) + list(model.session_emb.parameters())
    remaining_params = [
        p for n, p in model.named_parameters() if "unit_emb" not in n and "session_emb" not in n
    ]
    optimizer = SparseLamb(
        [
            {"params": special_emb_params, "sparse": True},
            {"params": remaining_params},
        ],
        lr=POYO_LR,
        weight_decay=POYO_WEIGHT_DECAY,
    )
    history = []
    best_state = None
    best_val = -np.inf
    criterion = nn.MSELoss(reduction="none")

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss_total = 0.0
        train_preds, train_targets = [], []
        for batch in train_loader:
            model_inputs = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch["model_inputs"].items()}
            target_values = batch["target_values"].to(DEVICE)
            target_weights = batch["target_weights"].to(DEVICE)
            output_mask = batch["model_inputs"]["output_mask"].to(DEVICE)
            output = model(**model_inputs)
            masked_output = output[output_mask]
            masked_target = target_values[output_mask]
            masked_weights = target_weights[output_mask]
            loss = criterion(masked_output, masked_target)
            loss = (loss * masked_weights.unsqueeze(-1)).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss_total += loss.item() * target_values.shape[0]
            train_preds.append((masked_output.detach().cpu().numpy() * 20.0).reshape(-1, 2))
            train_targets.append((masked_target.detach().cpu().numpy() * 20.0).reshape(-1, 2))
        train_preds_np = np.concatenate(train_preds, axis=0)
        train_targets_np = np.concatenate(train_targets, axis=0)
        train_metrics = r2_metrics(train_targets_np, train_preds_np)
        train_metrics["loss"] = train_loss_total / len(train_loader.dataset)

        val_metrics, _, _ = evaluate_poyo_loader(model, val_loader, denorm_std=20.0)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "val_loss": val_metrics["loss"],
                "train_r2_mean": train_metrics["r2_mean"],
                "val_r2_mean": val_metrics["r2_mean"],
            }
        )
        if val_metrics["r2_mean"] > best_val:
            best_val = val_metrics["r2_mean"]
            best_state = copy.deepcopy(model.state_dict())
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


def make_tensor_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    tensor_x = torch.from_numpy(X)
    tensor_y = torch.from_numpy(y)
    ds = TensorDataset(tensor_x, tensor_y)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def plot_learning_curve(history: List[Dict], title: str, out_path: Path) -> None:
    hist = pd.DataFrame(history)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(hist["epoch"], hist["train_loss"], label="train loss")
    ax.plot(hist["epoch"], hist["val_loss"], label="val loss")
    ax2 = ax.twinx()
    ax2.plot(hist["epoch"], hist["val_r2_mean"], color="green", label="val R^2")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax2.set_ylabel("Val R^2")
    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines + lines2, labels + labels2, loc="center right")
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)


def plot_pred_vs_actual(sample_df: pd.DataFrame, y_pred: np.ndarray, model_name: str, out_path: Path) -> None:
    plot_trial = int(sample_df["trial_id"].iloc[0])
    plot_df = sample_df.loc[sample_df["trial_id"] == plot_trial].copy().reset_index(drop=True)
    n = min(len(plot_df), len(y_pred), 150)
    t = plot_df["time"].to_numpy()[:n]
    y_true = plot_df[["vx", "vy"]].to_numpy()[:n]
    y_hat = y_pred[:n]
    fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
    axes[0].plot(t, y_true[:, 0], label="actual vx")
    axes[0].plot(t, y_hat[:, 0], label="pred vx", alpha=0.8)
    axes[0].legend(loc="upper right")
    axes[0].set_ylabel("vx (cm/s)")
    axes[1].plot(t, y_true[:, 1], label="actual vy")
    axes[1].plot(t, y_hat[:, 1], label="pred vy", alpha=0.8)
    axes[1].legend(loc="upper right")
    axes[1].set_ylabel("vy (cm/s)")
    axes[1].set_xlabel("Time (s)")
    fig.suptitle(f"{model_name}: predicted vs actual on one held-out trial")
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)


def main():
    set_seed(RANDOM_SEED)

    selected = download_smallest_valid_asset()
    structure = inspect_nwb_file(selected.local_path)
    session = load_session_arrays(selected.local_path)

    trial_df = choose_successful_trials(session["trials"], context_sec=CONTEXT_SEC)
    split_map = split_trials(trial_df, RANDOM_SEED)
    sample_df = build_sample_table(
        session["vel_timestamps"],
        session["vel_values"],
        trial_df,
        split_map,
        context_sec=CONTEXT_SEC,
        stride_sec=TARGET_STRIDE_SEC,
    )

    end_time = max(session["vel_timestamps"][-1], max(st[-1] for st in session["spike_times_by_unit"] if len(st)))
    binned_counts = bin_full_session(session["spike_times_by_unit"], end_time=end_time, bin_size_sec=BIN_SIZE_SEC)
    context_bins = int(round(CONTEXT_SEC / BIN_SIZE_SEC))

    split_frames = {
        split: sample_df.loc[sample_df["split"] == split].copy().reset_index(drop=True)
        for split in ["train", "val", "test"]
    }
    if FAST_DEV_RUN:
        for split, limit in FAST_DEV_MAX_SAMPLES.items():
            split_frames[split] = split_frames[split].head(limit).copy().reset_index(drop=True)

    X_train, y_train = build_baseline_arrays(split_frames["train"], binned_counts, context_bins)
    X_val, y_val = build_baseline_arrays(split_frames["val"], binned_counts, context_bins)
    X_test, y_test = build_baseline_arrays(split_frames["test"], binned_counts, context_bins)

    X_train_flat = X_train.reshape(len(X_train), -1)
    X_val_flat = X_val.reshape(len(X_val), -1)
    X_test_flat = X_test.reshape(len(X_test), -1)
    feat_mean = X_train_flat.mean(axis=0, keepdims=True)
    feat_std = X_train_flat.std(axis=0, keepdims=True) + 1e-6
    X_train_flat_n = (X_train_flat - feat_mean) / feat_std
    X_val_flat_n = (X_val_flat - feat_mean) / feat_std
    X_test_flat_n = (X_test_flat - feat_mean) / feat_std
    X_train_seq_n = X_train_flat_n.reshape(len(X_train), context_bins, X_train.shape[-1])
    X_val_seq_n = X_val_flat_n.reshape(len(X_val), context_bins, X_val.shape[-1])
    X_test_seq_n = X_test_flat_n.reshape(len(X_test), context_bins, X_test.shape[-1])
    y_mean = y_train.mean(axis=0, keepdims=True)
    y_std = y_train.std(axis=0, keepdims=True) + 1e-6
    y_train_n = ((y_train - y_mean) / y_std).astype(np.float32)
    y_val_n = ((y_val - y_mean) / y_std).astype(np.float32)
    y_test_n = ((y_test - y_mean) / y_std).astype(np.float32)

    results = {}

    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_flat_n, y_train)
    ridge_pred = ridge.predict(X_test_flat_n).astype(np.float32)
    results["Wiener"] = r2_metrics(y_test, ridge_pred)
    plot_pred_vs_actual(split_frames["test"], ridge_pred, "Wiener", ARTIFACTS_DIR / "wiener_pred_vs_actual.png")

    mlp = MLPDecoder(input_dim=X_train_flat_n.shape[1]).to(DEVICE)
    mlp_train_loader = make_tensor_loader(X_train_flat_n.astype(np.float32), y_train_n, MLP_BATCH_SIZE, True)
    mlp_val_loader = make_tensor_loader(X_val_flat_n.astype(np.float32), y_val_n, MLP_BATCH_SIZE, False)
    mlp_test_loader = make_tensor_loader(X_test_flat_n.astype(np.float32), y_test_n, MLP_BATCH_SIZE, False)
    mlp, mlp_hist = train_torch_decoder(
        mlp,
        mlp_train_loader,
        mlp_val_loader,
        MAX_EPOCHS_MLP,
        lr=3e-4,
        target_mean=y_mean,
        target_std=y_std,
    )
    mlp_test_metrics, mlp_pred, _ = iterate_batches(
        mlp_test_loader,
        mlp,
        optimizer=None,
        target_mean=y_mean,
        target_std=y_std,
    )
    results["MLP"] = {k: float(v) for k, v in mlp_test_metrics.items() if k.startswith("r2")}
    plot_learning_curve(mlp_hist, "MLP learning curve", ARTIFACTS_DIR / "mlp_learning_curve.png")
    plot_pred_vs_actual(split_frames["test"], mlp_pred, "MLP", ARTIFACTS_DIR / "mlp_pred_vs_actual.png")

    gru = GRUDecoder(input_dim=X_train_seq_n.shape[-1]).to(DEVICE)
    gru_train_loader = make_tensor_loader(X_train_seq_n.astype(np.float32), y_train_n, GRU_BATCH_SIZE, True)
    gru_val_loader = make_tensor_loader(X_val_seq_n.astype(np.float32), y_val_n, GRU_BATCH_SIZE, False)
    gru_test_loader = make_tensor_loader(X_test_seq_n.astype(np.float32), y_test_n, GRU_BATCH_SIZE, False)
    gru, gru_hist = train_torch_decoder(
        gru,
        gru_train_loader,
        gru_val_loader,
        MAX_EPOCHS_GRU,
        lr=3e-4,
        target_mean=y_mean,
        target_std=y_std,
    )
    gru_test_metrics, gru_pred, _ = iterate_batches(
        gru_test_loader,
        gru,
        optimizer=None,
        target_mean=y_mean,
        target_std=y_std,
    )
    results["GRU"] = {k: float(v) for k, v in gru_test_metrics.items() if k.startswith("r2")}
    plot_learning_curve(gru_hist, "GRU learning curve", ARTIFACTS_DIR / "gru_learning_curve.png")
    plot_pred_vs_actual(split_frames["test"], gru_pred, "GRU", ARTIFACTS_DIR / "gru_pred_vs_actual.png")

    readout_spec = MODALITY_REGISTRY["cursor_velocity_2d"]
    poyo = POYO(
        sequence_length=1.0,
        latent_step=0.125,
        num_latents_per_step=16,
        dim=64,
        depth=6,
        dim_head=64,
        cross_heads=2,
        self_heads=8,
        ffn_dropout=0.2,
        lin_dropout=0.4,
        atn_dropout=0.2,
        readout_spec=readout_spec,
    )
    session_id = Path(selected.path).stem.replace("_behavior+ecephys", "")
    poyo.unit_emb.initialize_vocab(session["unit_ids"])
    poyo.session_emb.initialize_vocab([session_id])
    train_ds = POYOSingleQueryDataset(split_frames["train"], session["spike_times_by_unit"], session["unit_ids"], session_id, poyo)
    val_ds = POYOSingleQueryDataset(split_frames["val"], session["spike_times_by_unit"], session["unit_ids"], session_id, poyo)
    test_ds = POYOSingleQueryDataset(split_frames["test"], session["spike_times_by_unit"], session["unit_ids"], session_id, poyo)
    train_loader = DataLoader(train_ds, batch_size=POYO_BATCH_SIZE, shuffle=True, collate_fn=tb_collate)
    val_loader = DataLoader(val_ds, batch_size=POYO_BATCH_SIZE, shuffle=False, collate_fn=tb_collate)
    test_loader = DataLoader(test_ds, batch_size=POYO_BATCH_SIZE, shuffle=False, collate_fn=tb_collate)
    poyo, poyo_hist = train_poyo_model(poyo, train_loader, val_loader, MAX_EPOCHS_POYO)
    poyo_test_metrics, poyo_pred, _ = evaluate_poyo_loader(poyo, test_loader, denorm_std=20.0)
    results["POYO"] = {k: float(v) for k, v in poyo_test_metrics.items() if k.startswith("r2")}
    plot_learning_curve(poyo_hist, "POYO learning curve", ARTIFACTS_DIR / "poyo_learning_curve.png")
    plot_pred_vs_actual(split_frames["test"], poyo_pred, "POYO", ARTIFACTS_DIR / "poyo_pred_vs_actual.png")

    comparison_df = pd.DataFrame(results).T.sort_index()
    comparison_df.to_csv(ARTIFACTS_DIR / "comparison_metrics.csv")

    with open(ARTIFACTS_DIR / "metrics.json", "w") as f:
        json.dump(
            {
                "selected_asset": selected.__dict__,
                "nwb_structure": structure,
                "results": results,
                "sample_counts": {k: int(len(v)) for k, v in split_frames.items()},
                "fast_dev_run": FAST_DEV_RUN,
            },
            f,
            indent=2,
            default=json_default,
        )

    with open(ARTIFACTS_DIR / "run_metadata.json", "w") as f:
        json.dump(
            {
                "device": DEVICE,
                "random_seed": RANDOM_SEED,
                "context_sec": CONTEXT_SEC,
                "bin_size_sec": BIN_SIZE_SEC,
                "target_stride_sec": TARGET_STRIDE_SEC,
                "max_epochs": {
                    "mlp": MAX_EPOCHS_MLP,
                    "gru": MAX_EPOCHS_GRU,
                    "poyo": MAX_EPOCHS_POYO,
                },
                "selected_asset": selected.__dict__,
            },
            f,
            indent=2,
            default=json_default,
        )

    print("Selected asset:", selected)
    print(comparison_df)
    print(f"Artifacts written to: {ARTIFACTS_DIR}")


# Part 3: DANDI Data Download / Stream + NWB Inspection

In [ ]:
set_seed(RANDOM_SEED)
selected = download_smallest_valid_asset(candidate_limit=MAX_CANDIDATES_TO_CHECK)
structure = inspect_nwb_file(selected.local_path)

provenance_df = pd.DataFrame(
    [
        {
            "dandiset_id": selected.dandiset_id,
            "published_version": selected.version_id,
            "asset_path": selected.path,
            "asset_size_mb": round(selected.size_bytes / 1e6, 3),
            "subject_id": selected.subject_id,
            "session_description": selected.session_description,
            "duration_sec": round(selected.duration_sec, 2),
            "units_count": selected.units_count,
            "trials_count": selected.trials_count,
            "local_path": str(selected.local_path),
        }
    ]
)
display(provenance_df)
print("Verified NWB structure:")
print(json.dumps(structure, indent=2, default=json_default))


In [ ]:
from pynwb import NWBHDF5IO

with NWBHDF5IO(str(selected.local_path), "r", load_namespaces=True) as io:
    nwb = io.read()
    print("Behavior interfaces:", list(nwb.processing["behavior"].data_interfaces.keys()))
    try:
        from nwbwidgets import nwb2widget
        display(nwb2widget(nwb))
    except Exception as exc:
        print("nwbwidgets could not render in this environment:", repr(exc))
        print("Falling back to the printed key inventory above.")


In [ ]:
session = load_session_arrays(selected.local_path)

def plot_spike_raster(spike_times_by_unit, max_units=12, t_start=0.0, t_end=10.0, out_path=Path("artifacts/spike_raster.png")):
    fig, ax = plt.subplots(figsize=(10, 4))
    for unit_idx, spike_times in enumerate(spike_times_by_unit[:max_units]):
        mask = (spike_times >= t_start) & (spike_times <= t_end)
        ax.vlines(spike_times[mask], unit_idx + 0.6, unit_idx + 1.4, linewidth=0.8)
    ax.set_title(f"Spike raster for the first {max_units} units")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Unit index")
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.show()
    plt.close(fig)

def plot_velocity_trace(timestamps, vel, t_start=0.0, t_end=10.0, out_path=Path("artifacts/cursor_velocity_trace.png")):
    mask = (timestamps >= t_start) & (timestamps <= t_end)
    fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
    axes[0].plot(timestamps[mask], vel[mask, 0])
    axes[0].set_ylabel("vx (cm/s)")
    axes[1].plot(timestamps[mask], vel[mask, 1])
    axes[1].set_ylabel("vy (cm/s)")
    axes[1].set_xlabel("Time (s)")
    fig.suptitle("Cursor velocity trace")
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.show()
    plt.close(fig)

plot_spike_raster(session["spike_times_by_unit"])
plot_velocity_trace(session["vel_timestamps"], session["vel_values"])


# Part 4: Preprocessing + Split Construction

In [ ]:
trial_df = choose_successful_trials(session["trials"], context_sec=CONTEXT_SEC)
split_map = split_trials(trial_df, RANDOM_SEED)
sample_df = build_sample_table(
    session["vel_timestamps"],
    session["vel_values"],
    trial_df,
    split_map,
    context_sec=CONTEXT_SEC,
    stride_sec=TARGET_STRIDE_SEC,
)

split_frames = {
    split: sample_df.loc[sample_df["split"] == split].copy().reset_index(drop=True)
    for split in ["train", "val", "test"]
}
if FAST_DEV_RUN:
    for split, limit in FAST_DEV_MAX_SAMPLES.items():
        split_frames[split] = split_frames[split].head(limit).copy().reset_index(drop=True)

split_summary = pd.DataFrame(
    {
        "num_trials": {split: len(split_map[split]) for split in split_map},
        "num_samples": {split: len(split_frames[split]) for split in split_frames},
        "trial_ids_head": {split: split_map[split][:5].tolist() for split in split_map},
    }
)
display(split_summary)

end_time = max(
    session["vel_timestamps"][-1],
    max(st[-1] for st in session["spike_times_by_unit"] if len(st)),
)
binned_counts = bin_full_session(
    session["spike_times_by_unit"],
    end_time=end_time,
    bin_size_sec=BIN_SIZE_SEC,
)
context_bins = int(round(CONTEXT_SEC / BIN_SIZE_SEC))

X_train, y_train = build_baseline_arrays(split_frames["train"], binned_counts, context_bins)
X_val, y_val = build_baseline_arrays(split_frames["val"], binned_counts, context_bins)
X_test, y_test = build_baseline_arrays(split_frames["test"], binned_counts, context_bins)

X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat = X_val.reshape(len(X_val), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

feat_mean = X_train_flat.mean(axis=0, keepdims=True)
feat_std = X_train_flat.std(axis=0, keepdims=True) + 1e-6
X_train_flat_n = (X_train_flat - feat_mean) / feat_std
X_val_flat_n = (X_val_flat - feat_mean) / feat_std
X_test_flat_n = (X_test_flat - feat_mean) / feat_std

X_train_seq_n = X_train_flat_n.reshape(len(X_train), context_bins, X_train.shape[-1])
X_val_seq_n = X_val_flat_n.reshape(len(X_val), context_bins, X_val.shape[-1])
X_test_seq_n = X_test_flat_n.reshape(len(X_test), context_bins, X_test.shape[-1])

y_mean = y_train.mean(axis=0, keepdims=True)
y_std = y_train.std(axis=0, keepdims=True) + 1e-6
y_train_n = ((y_train - y_mean) / y_std).astype(np.float32)
y_val_n = ((y_val - y_mean) / y_std).astype(np.float32)
y_test_n = ((y_test - y_mean) / y_std).astype(np.float32)

print("Feature tensor shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)


# Part 5: Baseline 1 - Wiener Filter

The Wiener filter baseline is a linear decoder: flatten the last `1s` of binned spike counts and learn a linear mapping to the current 2D cursor velocity. This is a classic neural decoding baseline because it is fast, interpretable, and provides a good sanity check before training nonlinear models.

In [ ]:
results = {}

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_flat_n, y_train)
wiener_pred = ridge.predict(X_test_flat_n).astype(np.float32)
results["Wiener"] = r2_metrics(y_test, wiener_pred)

plot_pred_vs_actual(split_frames["test"], wiener_pred, "Wiener", ARTIFACTS_DIR / "wiener_pred_vs_actual.png")
display(pd.DataFrame([results["Wiener"]], index=["Wiener"]))


# Part 6: Baseline 2 - MLP

The MLP baseline uses the same `1s` spike-count history as the Wiener filter, but replaces the linear readout with a small multilayer perceptron. This lets us test whether a simple nonlinear function of the same binned features already closes part of the gap to POYO.

In [ ]:
mlp = MLPDecoder(input_dim=X_train_flat_n.shape[1]).to(DEVICE)
mlp_train_loader = make_tensor_loader(X_train_flat_n.astype(np.float32), y_train_n, MLP_BATCH_SIZE, True)
mlp_val_loader = make_tensor_loader(X_val_flat_n.astype(np.float32), y_val_n, MLP_BATCH_SIZE, False)
mlp_test_loader = make_tensor_loader(X_test_flat_n.astype(np.float32), y_test_n, MLP_BATCH_SIZE, False)

mlp, mlp_hist = train_torch_decoder(
    mlp,
    mlp_train_loader,
    mlp_val_loader,
    MAX_EPOCHS_MLP,
    lr=3e-4,
    target_mean=y_mean,
    target_std=y_std,
)
mlp_test_metrics, mlp_pred, _ = iterate_batches(
    mlp_test_loader,
    mlp,
    optimizer=None,
    target_mean=y_mean,
    target_std=y_std,
)
results["MLP"] = {k: float(v) for k, v in mlp_test_metrics.items() if k.startswith("r2")}

plot_learning_curve(mlp_hist, "MLP learning curve", ARTIFACTS_DIR / "mlp_learning_curve.png")
plot_pred_vs_actual(split_frames["test"], mlp_pred, "MLP", ARTIFACTS_DIR / "mlp_pred_vs_actual.png")
display(pd.DataFrame([results["MLP"]], index=["MLP"]))


# Part 7: Baseline 3 - GRU

The GRU baseline keeps the `1s` history as an explicit sequence instead of flattening it. This is a closer sequential baseline for POYO because the model can accumulate information over the spike-count bins instead of seeing them only as a static vector.

In [ ]:
gru = GRUDecoder(input_dim=X_train_seq_n.shape[-1]).to(DEVICE)
gru_train_loader = make_tensor_loader(X_train_seq_n.astype(np.float32), y_train_n, GRU_BATCH_SIZE, True)
gru_val_loader = make_tensor_loader(X_val_seq_n.astype(np.float32), y_val_n, GRU_BATCH_SIZE, False)
gru_test_loader = make_tensor_loader(X_test_seq_n.astype(np.float32), y_test_n, GRU_BATCH_SIZE, False)

gru, gru_hist = train_torch_decoder(
    gru,
    gru_train_loader,
    gru_val_loader,
    MAX_EPOCHS_GRU,
    lr=3e-4,
    target_mean=y_mean,
    target_std=y_std,
)
gru_test_metrics, gru_pred, _ = iterate_batches(
    gru_test_loader,
    gru,
    optimizer=None,
    target_mean=y_mean,
    target_std=y_std,
)
results["GRU"] = {k: float(v) for k, v in gru_test_metrics.items() if k.startswith("r2")}

plot_learning_curve(gru_hist, "GRU learning curve", ARTIFACTS_DIR / "gru_learning_curve.png")
plot_pred_vs_actual(split_frames["test"], gru_pred, "GRU", ARTIFACTS_DIR / "gru_pred_vs_actual.png")
display(pd.DataFrame([results["GRU"]], index=["GRU"]))


# Part 8: POYO Single-Session Training Using the Official Implementation

POYO differs from the baselines in two important ways:

1. It treats **individual spikes as tokens** instead of first binning them into firing rates.
2. It uses a **Perceiver-style latent bottleneck** with time-aware attention, so the model can compress a variable number of spikes in a `1s` context window before decoding velocity.

Below, the educational diagram is generated directly in Python so the notebook stays self-contained.


In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_poyo_diagram(out_path=Path("artifacts/poyo_architecture_diagram.png")):
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    def box(x, y, w, h, text, color):
        patch = FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.02,rounding_size=0.02",
            linewidth=1.5,
            facecolor=color,
            edgecolor="black",
        )
        ax.add_patch(patch)
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=11)

    def arrow(x0, y0, x1, y1):
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="->", mutation_scale=12, linewidth=1.5))

    box(0.03, 0.22, 0.18, 0.56, "Spike tokens\n(unit id + timestamp)", "#d8ecff")
    box(0.29, 0.22, 0.18, 0.56, "Cross-attention\ninto latent tokens", "#dff5dd")
    box(0.55, 0.22, 0.18, 0.56, "Latent self-attention\nprocessing stack", "#fff0c9")
    box(0.81, 0.22, 0.16, 0.56, "Query at output time\n-> velocity", "#ffd9d9")
    arrow(0.21, 0.50, 0.29, 0.50)
    arrow(0.47, 0.50, 0.55, 0.50)
    arrow(0.73, 0.50, 0.81, 0.50)
    ax.set_title("POYO in one sentence: spikes become tokens, latents summarize them, queries read out velocity")
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.show()
    plt.close(fig)

draw_poyo_diagram()


In [ ]:
readout_spec = MODALITY_REGISTRY["cursor_velocity_2d"]
poyo = POYO(
    sequence_length=1.0,
    latent_step=0.125,
    num_latents_per_step=16,
    dim=64,
    depth=6,
    dim_head=64,
    cross_heads=2,
    self_heads=8,
    ffn_dropout=0.2,
    lin_dropout=0.4,
    atn_dropout=0.2,
    readout_spec=readout_spec,
)

session_id = Path(selected.path).stem.replace("_behavior+ecephys", "")
poyo.unit_emb.initialize_vocab(session["unit_ids"])
poyo.session_emb.initialize_vocab([session_id])

train_ds = POYOSingleQueryDataset(
    split_frames["train"],
    session["spike_times_by_unit"],
    session["unit_ids"],
    session_id,
    poyo,
)
val_ds = POYOSingleQueryDataset(
    split_frames["val"],
    session["spike_times_by_unit"],
    session["unit_ids"],
    session_id,
    poyo,
)
test_ds = POYOSingleQueryDataset(
    split_frames["test"],
    session["spike_times_by_unit"],
    session["unit_ids"],
    session_id,
    poyo,
)

train_loader = DataLoader(train_ds, batch_size=POYO_BATCH_SIZE, shuffle=True, collate_fn=tb_collate)
val_loader = DataLoader(val_ds, batch_size=POYO_BATCH_SIZE, shuffle=False, collate_fn=tb_collate)
test_loader = DataLoader(test_ds, batch_size=POYO_BATCH_SIZE, shuffle=False, collate_fn=tb_collate)

poyo, poyo_hist = train_poyo_model(poyo, train_loader, val_loader, MAX_EPOCHS_POYO)
poyo_test_metrics, poyo_pred, _ = evaluate_poyo_loader(poyo, test_loader, denorm_std=20.0)
results["POYO"] = {k: float(v) for k, v in poyo_test_metrics.items() if k.startswith("r2")}

plot_learning_curve(poyo_hist, "POYO learning curve", ARTIFACTS_DIR / "poyo_learning_curve.png")
plot_pred_vs_actual(split_frames["test"], poyo_pred, "POYO", ARTIFACTS_DIR / "poyo_pred_vs_actual.png")
display(pd.DataFrame([results["POYO"]], index=["POYO"]))


# Part 9: Evaluation + Comparisons

This section aggregates all four models into a single comparison table and saves the final metrics under `./artifacts/metrics.json`. The paper's Section 3.2 reports an average single-session `R^2 ≈ 0.84` on random-target sessions and a higher average on center-out sessions, so the fairest paper reference for our auto-selected asset is the random-target regime.

In [ ]:
comparison_df = pd.DataFrame(results).T.loc[["Wiener", "MLP", "GRU", "POYO"]]
display(comparison_df)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(comparison_df.index, comparison_df["r2_mean"], color=["#8db1ff", "#ffbe7a", "#7ecf9a", "#ff8f8f"])
ax.axhline(0.8402, color="black", linestyle="--", linewidth=1.2, label="POYO paper RT average (Section 3.2)")
ax.set_ylabel("Test R^2")
ax.set_title("Model comparison on the held-out split")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "comparison_bar_chart.png", dpi=180)
plt.show()
plt.close(fig)

run_metadata = {
    "fast_dev_run": FAST_DEV_RUN,
    "random_seed": RANDOM_SEED,
    "context_sec": CONTEXT_SEC,
    "bin_size_sec": BIN_SIZE_SEC,
    "target_stride_sec": TARGET_STRIDE_SEC,
    "max_epochs": {
        "mlp": MAX_EPOCHS_MLP,
        "gru": MAX_EPOCHS_GRU,
        "poyo": MAX_EPOCHS_POYO,
    },
    "device": DEVICE,
    "torch_brain_clone_commit": clone_commit,
    "torch_brain_version": getattr(torch_brain, "__version__", "unknown"),
    "selected_asset": selected.__dict__,
    "nwb_structure": structure,
    "sample_counts": {split: int(len(df)) for split, df in split_frames.items()},
}

with open(ARTIFACTS_DIR / "metrics.json", "w") as f:
    json.dump(
        {
            "results": results,
            "selected_asset": selected.__dict__,
            "sample_counts": {split: int(len(df)) for split, df in split_frames.items()},
            "paper_reference": {
                "proposal_target": "~0.97 on an easier center-out setting",
                "paper_single_session_rt_average_r2": 0.8402,
            },
        },
        f,
        indent=2,
        default=json_default,
    )

with open(ARTIFACTS_DIR / "run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=2, default=json_default)

print("Saved metrics to:", ARTIFACTS_DIR / "metrics.json")
print("Saved run metadata to:", ARTIFACTS_DIR / "run_metadata.json")


# Part 10: Repro Notes

In [ ]:
repro_df = pd.DataFrame(
    [
        {
            "device": DEVICE,
            "fast_dev_run": FAST_DEV_RUN,
            "random_seed": RANDOM_SEED,
            "context_sec": CONTEXT_SEC,
            "bin_size_sec": BIN_SIZE_SEC,
            "target_stride_sec": TARGET_STRIDE_SEC,
            "clone_commit": clone_commit,
            "asset_path": selected.path,
            "asset_size_mb": round(selected.size_bytes / 1e6, 3),
            "note": "Best-effort single-session reproduction on the smallest valid published session.",
        }
    ]
)
display(repro_df)

print("Known limitations:")
print("- The notebook auto-selects the smallest valid session, which may not match the exact session distribution used in the paper averages.")
print("- FAST_DEV_RUN trades off accuracy for speed; switch it off for a stronger best-effort Colab run.")
print("- The paper uses richer weighting and evaluation intervals for the Perich/Miller datasets; here we keep the split faithful and the runtime practical for a single Colab session.")
